In [2]:
import joblib
import pandas as pd
import numpy as np

# Carrega os arquivos de modelo 
modelo = joblib.load('modelo_copa_2026.pkl')
scaler = joblib.load('scaler_copa_2026.pkl')

print("Modelo e Scaler carregados com sucesso! ")

Modelo e Scaler carregados com sucesso! 


In [3]:
# Função de previsao dos jogos

def prever_confronto(peso_torneio, delta_ranking, gols_man, gols_vis, conf_totais, vit_man, vit_vis, nome_man='Mandante', nome_vis='Visitante'):
    """"
    Recebe os atributos táticos das duas equipes, passa pelo Saclaer e devolve a probabilidade exata do placar (Vitoria Mandante, Empate, Vitoria Visitante)
     
    """
    # Deixar os dados no mesmo formato do treino
    dados_confronto = pd.DataFrame([{
        'PESO_COMPETICAO': peso_torneio,
        'DELTA_RANKING_PONTOS': delta_ranking,
        'MED_GOLS_MARCADOS_MANDANTE': gols_man,
        'MED_GOLS_MARCADOS_VISITANTE': gols_vis,
        'HISTORICO_CONFRONTOS_TOTAIS': conf_totais,
        'HISTORICO_VITORIAS_MANDANTE': vit_man,
        'HISTORICO_VITORIAS_VISITANTE': vit_vis       
    }])
    
    dados_scaled = scaler.transform(dados_confronto)
    
    # probabilidade para cada classe
    probs = modelo.predict_proba(dados_scaled)[0]
    
    print(f"🔮 Previsão: {nome_man} vs {nome_vis}")
    print(f"   -> Probabilidade de Vitória do {nome_man}: {round(probs[2] * 100, 2)}%")
    print(f"   -> Probabilidade de Empate: {round(probs[1] * 100, 2)}%")
    print(f"   -> Probabilidade de Vitória do {nome_vis}: {round(probs[0] * 100, 2)}%")
    print("-" * 50)
    
    return probs

In [4]:
# Executa o teste do motor preditor
probabilidades = prever_confronto(
    peso_torneio=3,     # Copa do Mundo
    delta_ranking=150.0, # Brasil superior no ranking
    gols_man=2.1,       # Média de gols do Brasil nos últimos 24 meses
    gols_vis=1.4,       # Média de gols da Itália nos últimos 24 meses
    conf_totais=5,      # Jogos históricos entre eles
    vit_man=3,          # Vitórias históricas do Brasil
    vit_vis=1,          # Vitórias históricas da Itália
    nome_man="Brasil",
    nome_vis="Itália"
)

🔮 Previsão: Brasil vs Itália
   -> Probabilidade de Vitória do Brasil: 64.85%
   -> Probabilidade de Empate: 17.46%
   -> Probabilidade de Vitória do Itália: 17.69%
--------------------------------------------------


In [5]:
import sqlalchemy

SERVER = 'FELIPE-PC\\SQLEXPRESS'
DATABASE = 'DB_COPA_2026'
connection_url = f"mssql+pyodbc://@{SERVER}/{DATABASE}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = sqlalchemy.create_engine(connection_url)

In [6]:
query_perfis = """
WITH CTE_Gols_Recentes AS (
    -- Consolida a média de gols dos últimos 24 meses
    SELECT 
        ID_SELECAO,
        AVG(GOLS_MARCADOS) AS MED_GOLS_RECENTE
    FROM (
        SELECT ID_SELECAO_MANDANTE AS ID_SELECAO, GOLS_MANDANTE AS GOLS_MARCADOS, DATA_PARTIDA FROM fato_partidas
        UNION ALL
        SELECT ID_SELECAO_VISITANTE AS ID_SELECAO, GOLS_VISITANTE AS GOLS_MARCADOS, DATA_PARTIDA FROM fato_partidas
    ) AS todas_partidas
    WHERE DATA_PARTIDA >= DATEADD(MONTH, -24, (SELECT MAX(DATA_PARTIDA) FROM fato_partidas))
    GROUP BY ID_SELECAO
),
CTE_Ultimo_Ranking AS (
    -- Mapeado com as colunas reais: 'team', '[total.points]' e 'date'
    SELECT 
        team,
        [total.points],
        ROW_NUMBER() OVER (PARTITION BY team ORDER BY date DESC) AS RN
    FROM stg_ranking_fifa
)
SELECT 
    s.ID_SELECAO,
    s.NOME_SELECAO,
    ISNULL(g.MED_GOLS_RECENTE, 0) AS MED_GOLS,
    ISNULL(r.[total.points], 1000) AS PONTOS_FIFA
FROM dim_selecoes s
LEFT JOIN CTE_Gols_Recentes g ON s.ID_SELECAO = g.ID_SELECAO
LEFT JOIN CTE_Ultimo_Ranking r ON s.NOME_SELECAO = r.team AND r.RN = 1;
"""

# Executa a carga com o mapeamento idêntico ao banco
df_perfis_selecoes = pd.read_sql(query_perfis, engine)
df_perfis_selecoes.set_index('NOME_SELECAO', inplace=True)

print(f"✅ Tabela de Consulta Automatizada criada! {df_perfis_selecoes.shape[0]} seleções tagueadas.")
df_perfis_selecoes.sample(10)

✅ Tabela de Consulta Automatizada criada! 364 seleções tagueadas.


,ID_SELECAO,MED_GOLS,PONTOS_FIFA
NOME_SELECAO,,,
Tahiti,313,1,1016.83
Åland Islands,3,0,1000.00
Romani people,252,0,1000.00
Croatia,80,2,1701.31
Italy,155,1,1714.29
North Korea,221,1,1000.00
Republic of Ireland,248,1,1403.84
Occitania,228,0,1000.00
Latvia,174,0,1095.98


In [7]:

print(pd.read_sql("SELECT TOP 1 * FROM stg_ranking_fifa", engine).columns.tolist())

['date', 'semester', 'rank', 'team', 'acronym', 'total.points', 'previous.points', 'diff.points']


In [8]:
import json

# Os 12 grupos oficiais e reais da Copa do Mundo de 2026
dados_dos_grupos = {
    "Grupo A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "Grupo B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "Grupo C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "Grupo D": ["USA", "Paraguay", "Australia", "Türkiye"],
    "Grupo E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "Grupo F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "Grupo G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "Grupo H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "Grupo I": ["France", "Senegal", "Iraq", "Norway"],
    "Grupo J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "Grupo K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "Grupo L": ["England", "Croatia", "Ghana", "Panama"]
}

# Salva os dados em um arquivo JSON externo
with open('grupos_copa_2026.json', 'w', encoding='utf-8') as f:
    json.dump(dados_dos_grupos, f, ensure_ascii=False, indent=4)

print("💾 Arquivo de configuração 'grupos_copa_2026.json' gerado com sucesso!")

💾 Arquivo de configuração 'grupos_copa_2026.json' gerado com sucesso!


In [9]:
# O Python lê o arquivo externo de forma limpa e profissional
with open('grupos_copa_2026.json', 'r', encoding='utf-8') as f:
    grupos_copa = json.load(f)

# Exibe o chaveamento para garantir que funcionou
for grupo, times in grupos_copa.items():
    print(f"{grupo}: {times}")

Grupo A: ['Mexico', 'South Africa', 'South Korea', 'Czech Republic']
Grupo B: ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland']
Grupo C: ['Brazil', 'Morocco', 'Haiti', 'Scotland']
Grupo D: ['USA', 'Paraguay', 'Australia', 'Türkiye']
Grupo E: ['Germany', 'Curaçao', 'Ivory Coast', 'Ecuador']
Grupo F: ['Netherlands', 'Japan', 'Sweden', 'Tunisia']
Grupo G: ['Belgium', 'Egypt', 'Iran', 'New Zealand']
Grupo H: ['Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay']
Grupo I: ['France', 'Senegal', 'Iraq', 'Norway']
Grupo J: ['Argentina', 'Algeria', 'Austria', 'Jordan']
Grupo K: ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia']
Grupo L: ['England', 'Croatia', 'Ghana', 'Panama']


In [10]:
def buscar_historico_confronto(nome_man, nome_vis):
    """
    Conecta ao banco de dados e recupera o histórico real de confrontos diretos
    entre as duas seleções utilizando a view que criamos no SQL Server.
    """
    query = f"""
    SELECT 
        HISTORICO_CONFRONTOS_TOTAIS,
        HISTORICO_VITORIAS_MANDANTE,
        HISTORICO_VITORIAS_VISITANTE
    FROM vw_confronto_direto
    WHERE ID_SELECAO_MANDANTE = (SELECT ID_SELECAO FROM dim_selecoes WHERE NOME_SELECAO = '{nome_man}')
      AND ID_SELECAO_VISITANTE = (SELECT ID_SELECAO FROM dim_selecoes WHERE NOME_SELECAO = '{nome_vis}')
    """
    try:
        df_h2h = pd.read_sql(query, engine)
        if not df_h2h.empty:
            return df_h2h.iloc[0].to_dict()
    except:
        pass
    
    # Se as seleções nunca se enfrentaram na história, retorna o dicionário zerado
    return {'HISTORICO_CONFRONTOS_TOTAIS': 0, 'HISTORICO_VITORIAS_MANDANTE': 0, 'HISTORICO_VITORIAS_VISITANTE': 0}

In [72]:
import itertools

# ⚙️ PARÂMETRO MESTRE: Defina 'estocastico' para emoção/zebras ou 'deterministico' para a lógica pura da IA
MODO_SIMULACAO = 'deterministico' 

classificacao_geral = {}
print(f"🏃‍♂️ Inicializando a Fase de Grupos 2026 no modo: [{MODO_SIMULACAO.upper()}]...\n")

for nome_grupo, times in grupos_copa.items():
    # Inicializa a tabela do grupo zerada
    tabela = pd.DataFrame(index=times, columns=['P', 'J', 'V', 'E', 'D', 'GP', 'GC', 'SG'])
    tabela.fillna(0, inplace=True)
    
    # Todos contra todos dentro do grupo
    confrontos = list(itertools.combinations(times, 2))
    
    for time_man, time_vis in confrontos:
        # Coleta os dados da nossa Lookup Table e do Banco
        perfis = df_perfis_selecoes.loc[[time_man, time_vis]]
        med_gols_man = perfis.loc[time_man, 'MED_GOLS']
        med_gols_vis = perfis.loc[time_vis, 'MED_GOLS']
        delta_ranking = perfis.loc[time_man, 'PONTOS_FIFA'] - perfis.loc[time_vis, 'PONTOS_FIFA']
        
        h2h = buscar_historico_confronto(time_man, time_vis)
        
        # Prepara os dados para o modelo
        dados_confronto = pd.DataFrame([{
            'PESO_COMPETICAO': 3,
            'DELTA_RANKING_PONTOS': delta_ranking,
            'MED_GOLS_MARCADOS_MANDANTE': med_gols_man,
            'MED_GOLS_MARCADOS_VISITANTE': med_gols_vis,
            'HISTORICO_CONFRONTOS_TOTAIS': h2h['HISTORICO_CONFRONTOS_TOTAIS'],
            'HISTORICO_VITORIAS_MANDANTE': h2h['HISTORICO_VITORIAS_MANDANTE'],
            'HISTORICO_VITORIAS_VISITANTE': h2h['HISTORICO_VITORIAS_VISITANTE']
        }])
        
        # Predição de Probabilidades
        dados_scaled = scaler.transform(dados_confronto)
        probs = modelo.predict_proba(dados_scaled)[0] # [Classe 0: Vis, Classe 1: Empate, Classe 2: Man]
        
        # 🧠 A GRANDE VIRADA: Decisão do resultado baseado no modo escolhido
        if MODO_SIMULACAO == 'deterministico':
            # Abordagem Absoluta: O resultado de maior probabilidade sempre ganha (Sem sorteio)
            resultado_final = np.argmax(probs) 
            
            # Gols calculados friamente pela média arredondada
            gols_man_sim = int(round(med_gols_man))
            gols_vis_sim = int(round(med_gols_vis))
        else:
            # Abordagem Estocástica: Sorteio ponderado (Gira a roleta da IA)
            resultado_final = np.random.choice([0, 1, 2], p=probs)
            
            # Gols calculados via distribuição estatística de Poisson (Gera variabilidade)
            gols_man_sim = int(np.random.poisson(med_gols_man))
            gols_vis_sim = int(np.random.poisson(med_gols_vis))
        
        # ⚖️ Ajuste Fino dos Gols: Garante que o placar reflete o resultado decretado
        if resultado_final == 2 and gols_man_sim <= gols_vis_sim: # Vitória Mandante
            gols_man_sim = gols_vis_sim + 1
        elif resultado_final == 0 and gols_vis_sim <= gols_man_sim: # Vitória Visitante
            gols_vis_sim = gols_man_sim + 1
        elif resultado_final == 1: # Empate
            gols_man_sim = gols_vis_sim = max(gols_man_sim, gols_vis_sim)
            
        # 📊 Atualização de estatísticas na tabela
        tabela.loc[time_man, 'J'] += 1
        tabela.loc[time_vis, 'J'] += 1
        tabela.loc[time_man, 'GP'] += gols_man_sim
        tabela.loc[time_man, 'GC'] += gols_vis_sim
        tabela.loc[time_vis, 'GP'] += gols_vis_sim
        tabela.loc[time_vis, 'GC'] += gols_man_sim
        
        if resultado_final == 2:
            tabela.loc[time_man, 'P'] += 3
            tabela.loc[time_man, 'V'] += 1
            tabela.loc[time_vis, 'D'] += 1
        elif resultado_final == 0:
            tabela.loc[time_vis, 'P'] += 3
            tabela.loc[time_vis, 'V'] += 1
            tabela.loc[time_man, 'D'] += 1
        else:
            tabela.loc[time_man, 'P'] += 1
            tabela.loc[time_vis, 'P'] += 1
            tabela.loc[time_man, 'E'] += 1
            tabela.loc[time_vis, 'E'] += 1
            
        tabela['SG'] = tabela['GP'] - tabela['GC']

    # Critérios de Desempate oficiais da FIFA
    classificacao_geral[nome_grupo] = tabela.sort_values(by=['P', 'V', 'SG', 'GP'], ascending=False)

print("🏆 Fase de Grupos processada com sucesso!")

🏃‍♂️ Inicializando a Fase de Grupos 2026 no modo: [DETERMINISTICO]...

🏆 Fase de Grupos processada com sucesso!


In [62]:
# Inicializa as listas para separar os classificados
classificados_diretos = []
repescagem_terceiros = []

for nome_grupo, tabela in classificacao_geral.items():
    # Os dois primeiros de cada grupo garantem vaga direta
    classificados_diretos.append(tabela.index[0])
    classificados_diretos.append(tabela.index[1])
    
    # O terceiro colocado vai para a bacia das estatísticas da repescagem
    dados_terceiro = tabela.iloc[2].to_dict()
    dados_terceiro['TIME'] = tabela.index[2]
    dados_terceiro['GRUPO'] = nome_grupo
    repescagem_terceiros.append(dados_terceiro)

# Transforma a lista de terceiros em um DataFrame para aplicação das regras de desempate
df_terceiros_unificado = pd.DataFrame(repescagem_terceiros)

# Critério de desempate oficial da FIFA: Pontos, Vitórias, Saldo de Gols e Gols Pró
df_terceiros_ordenado = df_terceiros_unificado.sort_values(
    by=['P', 'V', 'SG', 'GP'], 
    ascending=False
).reset_index(drop=True)

# Captura os 8 primeiros colocados da tabela unificada de terceiros
melhores_terceiros = df_terceiros_ordenado.head(8)['TIME'].tolist()

# Une os 24 classificados diretos com os 8 repescados
lista_final_mata_mata = classificados_diretos + melhores_terceiros

print("📊 --- RANKING UNIFICADO DOS TERCEIROS COLOCADOS ---")
print(df_terceiros_ordenado[['TIME', 'GRUPO', 'P', 'V', 'SG', 'GP']])
print("-" * 60)
print(f"🚨 FASE DE GRUPOS ENCERRADA! {len(lista_final_mata_mata)} seleções carimbaram o passaporte para o Mata-Mata.")

📊 --- RANKING UNIFICADO DOS TERCEIROS COLOCADOS ---
              TIME    GRUPO  P  V  SG  GP
0            Haiti  Grupo C  3  1  -1   6
1           Sweden  Grupo F  3  1  -1   6
2          Austria  Grupo J  3  1  -1   6
3   Czech Republic  Grupo A  3  1  -1   4
4            Qatar  Grupo B  3  1  -1   4
5          Ecuador  Grupo E  3  1  -1   4
6            Egypt  Grupo G  3  1  -1   4
7          Uruguay  Grupo H  3  1  -1   4
8           France  Grupo I  3  1  -1   4
9       Uzbekistan  Grupo K  3  1  -1   4
10           Ghana  Grupo L  3  1  -1   4
11             USA  Grupo D  3  1  -1   1
------------------------------------------------------------
🚨 FASE DE GRUPOS ENCERRADA! 32 seleções carimbaram o passaporte para o Mata-Mata.


In [63]:
# Rodadas do mata amata

def simular_rodada_mata_mata(times_participantes, nome_rodada):
    """
    Simula uma rodada eliminatória pura. 
    Recebe uma lista de times, faz os confrontos em pares e retorna os vencedores.
    """
    print(f"⚔️ Iniciando: {nome_rodada} ({len(times_participantes)} seleções) ---")
    vencedores = []
    
    # Organiza os confrontos em pares (1º vs 2º, 3º vs 4º...)
    for i in range(0, len(times_participantes), 2):
        time_man = times_participantes[i]
        time_vis = times_participantes[i+1]
        
        # Coleta dados da Lookup Table
        perfis = df_perfis_selecoes.loc[[time_man, time_vis]]
        med_gols_man = perfis.loc[time_man, 'MED_GOLS']
        med_gols_vis = perfis.loc[time_vis, 'MED_GOLS']
        p_man = perfis.loc[time_man, 'PONTOS_FIFA']
        p_vis = perfis.loc[time_vis, 'PONTOS_FIFA']
        
        delta_ranking = p_man - p_vis
        h2h = buscar_historico_confronto(time_man, time_vis)
        
        # Prepara dados para a IA
        dados_confronto = pd.DataFrame([{
            'PESO_COMPETICAO': 3,
            'DELTA_RANKING_PONTOS': delta_ranking,
            'MED_GOLS_MARCADOS_MANDANTE': med_gols_man,
            'MED_GOLS_MARCADOS_VISITANTE': med_gols_vis,
            'HISTORICO_CONFRONTOS_TOTAIS': h2h['HISTORICO_CONFRONTOS_TOTAIS'],
            'HISTORICO_VITORIAS_MANDANTE': h2h['HISTORICO_VITORIAS_MANDANTE'],
            'HISTORICO_VITORIAS_VISITANTE': h2h['HISTORICO_VITORIAS_VISITANTE']
        }])
        
        dados_scaled = scaler.transform(dados_confronto)
        probs = modelo.predict_proba(dados_scaled)[0]
        
        # Decisão do Resultado baseado no MODO_SIMULACAO ativo
        if MODO_SIMULACAO == 'deterministico':
            resultado_final = np.argmax(probs)
        else:
            resultado_final = np.random.choice([0, 1, 2], p=probs)
            
        # Tratamento de Regra de Negócio: Se der Empate (1), vai para os Pênaltis
        if resultado_final == 1:
            # Peso do ranking decide a probabilidade dos pênaltis
            peso_man = p_man / (p_man + p_vis)
            peso_vis = p_vis / (p_man + p_vis)
            
            vencedor_penaltis = np.random.choice([time_man, time_vis], p=[peso_man, peso_vis])
            print(f" Em jogo tenso, {time_man} vs {time_vis} empataram. [{vencedor_penaltis}] avança nos pênaltis!")
            vencedores.append(vencedor_penaltis)
            
        elif resultado_final == 2: # Vitória do Mandante
            print(f" Regular: {time_man} venceu {time_vis} e avançou.")
            vencedores.append(time_man)
        else: # Vitória do Visitante
            print(f" Regular: {time_vis} venceu {time_man} e avançou.")
            vencedores.append(time_vis)
            
    print(f"✅ {nome_rodada} finalizado! {len(vencedores)} times qualificados.\n")
    return vencedores

# Executa o Round of 32 usando a nossa lista_final_mata_mata
oitavas_de_final_times = simular_rodada_mata_mata(lista_final_mata_mata, "Dezesseis-avos de Final (Round of 32)")

⚔️ Iniciando: Dezesseis-avos de Final (Round of 32) (32 seleções) ---
 Regular: Mexico venceu South Africa e avançou.
 Regular: Canada venceu Bosnia and Herzegovina e avançou.
 Regular: Morocco venceu Brazil e avançou.
 Regular: Paraguay venceu Australia e avançou.
 Regular: Germany venceu Curaçao e avançou.
 Regular: Netherlands venceu Japan e avançou.
 Regular: Belgium venceu New Zealand e avançou.
 Regular: Spain venceu Saudi Arabia e avançou.
 Regular: Senegal venceu Norway e avançou.
 Regular: Argentina venceu Algeria e avançou.
 Regular: Portugal venceu Colombia e avançou.
 Regular: England venceu Croatia e avançou.
 Regular: Haiti venceu Sweden e avançou.
 Regular: Austria venceu Czech Republic e avançou.
 Regular: Qatar venceu Ecuador e avançou.
 Regular: Egypt venceu Uruguay e avançou.
✅ Dezesseis-avos de Final (Round of 32) finalizado! 16 times qualificados.



In [73]:
import numpy as np
import pandas as pd

# ==============================================================================
# 1. MOTOR DE INFERÊNCIA ISOLADO (Previsão de Jogo Único)
# ==============================================================================
def prever_jogo_unico(time_man, time_vis, modo=MODO_SIMULACAO, peso_competicao=3):
    """
    Consulta a IA e retorna o resultado estruturado de um único confronto.
    Pode ser chamada isoladamente para qualquer par de seleções no seu projeto.
    """
    perfis = df_perfis_selecoes.loc[[time_man, time_vis]]
    med_gols_man = perfis.loc[time_man, 'MED_GOLS']
    med_gols_vis = perfis.loc[time_vis, 'MED_GOLS']
    delta_ranking = perfis.loc[time_man, 'PONTOS_FIFA'] - perfis.loc[time_vis, 'PONTOS_FIFA']
    
    h2h = buscar_historico_confronto(time_man, time_vis)
    
    dados_confronto = pd.DataFrame([{
        'PESO_COMPETICAO': peso_competicao,
        'DELTA_RANKING_PONTOS': delta_ranking,
        'MED_GOLS_MARCADOS_MANDANTE': med_gols_man,
        'MED_GOLS_MARCADOS_VISITANTE': med_gols_vis,
        'HISTORICO_CONFRONTOS_TOTAIS': h2h['HISTORICO_CONFRONTOS_TOTAIS'],
        'HISTORICO_VITORIAS_MANDANTE': h2h['HISTORICO_VITORIAS_MANDANTE'],
        'HISTORICO_VITORIAS_VISITANTE': h2h['HISTORICO_VITORIAS_VISITANTE']
    }])
    
    dados_scaled = scaler.transform(dados_confronto)
    probs = modelo.predict_proba(dados_scaled)[0]
    
    if modo == 'deterministico':
        resultado = np.argmax(probs)
    else:
        resultado = np.random.choice([0, 1, 2], p=probs)
        
    return resultado, perfis.loc[time_man, 'PONTOS_FIFA'], perfis.loc[time_vis, 'PONTOS_FIFA']


# ==============================================================================
# 2. MOTOR DO MATA-MATA RECURSIVO (Roda qualquer fase eliminatória)
# ==============================================================================
def simular_arvore_eliminatoria(lista_32_times, modo=MODO_SIMULACAO):
    """
    Executa a árvore completa do mata-mata até a grande final.
    Aceita tanto a lista vinda da fase de grupos quanto uma lista customizada pelo usuário.
    """
    fases = {
        "Dezesseis-avos de Final (Round of 32)": lista_32_times,
        "Oitavas de Final": [],
        "Quartas de Final": [],
        "Semifinal": [],
        "Grande Final": []
    }
    
    fases_ordem = ["Dezesseis-avos de Final (Round of 32)", "Oitavas de Final", "Quartas de Final", "Semifinal"]
    
    for atual in fases_ordem:
        times_da_fase = fases[atual]
        proxima_fase_nome = list(fases.keys())[list(fases.keys()).index(atual) + 1]
        
        print(f"\n🏆 --- SIMULANDO: {atual.upper()} ---")
        
        # Agrupa em pares consecutivos para simular o chaveamento fixo da árvore
        for i in range(0, len(times_da_fase), 2):
            t1, t2 = times_da_fase[i], times_da_fase[i+1]
            
            res, p1, p2 = prever_jogo_unico(t1, t2, modo=modo)
            
            if res == 1: # Empate técnico -> Decisão por Pênaltis via peso do Ranking FIFA
                w_p1, w_p2 = p1 / (p1 + p2), p2 / (p1 + p2)
                vencedor = np.random.choice([t1, t2], p=[w_p1, w_p2])
                print(f" ⚖️ {t1} vs {t2} empataram no tempo regulamentar. [{vencedor}] avança nos pênaltis!")
            elif res == 2:
                vencedor = t1
                print(f" 🟢 {t1} derrotou {t2} no tempo normal.")
            else:
                vencedor = t2
                print(f" 🟢 {t2} derrotou {t1} no tempo normal.")
                
            fases[proxima_fase_nome].append(vencedor)
            
    # Execução isolada da Grande Final
    finais = fases["Grande Final"]
    print(f"\n🔥 🏟️ --- A GRANDE FINAL DA COPA DO MUNDO: {finais[0]} VS {finais[1]} ---")
    campeao_res, pc1, pc2 = prever_jogo_unico(finais[0], finais[1], modo=modo)
    
    if campeao_res == 1:
        w_pc1, w_pc2 = pc1 / (pc1 + pc2), pc2 / (pc1 + pc2)
        campeao = np.random.choice([finais[0], finais[1],], p=[w_pc1, w_pc2])
        print(f" 🏆 FIM DE JOGO! Nos pênaltis históricos, o [{campeao.upper()}] É O CAMPEÃO DO MUNDO DE 2026!")
    elif campeao_res == 2:
        campeao = finais[0]
        print(f" 🏆 FIM DE JOGO! O [{campeao.upper()}] atropela o adversário e sagra-se CAMPEÃO DO MUNDO!")
    else:
        campeao = finais[1]
        print(f" 🏆 FIM DE JOGO! O [{campeao.upper()}] cala o estádio e levanta a TAÇA DA COPA DO MUNDO!")
        
    return fases

In [74]:
# Teste de Mesa rápido: O usuário quer ver as chances de um clássico isolado
resultado, _, _ = prever_jogo_unico("Brazil", "Argentina", modo="estocastico")
# Retorna 2 para vitória do mandante, 1 empate, 0 visitante
print(f"Resultado bruto calculado pela IA: {resultado}")

Resultado bruto calculado pela IA: 2


In [75]:
# Lista customizada de 32 times estruturada em pares de confronto
meu_mata_mata_customizado = [
    "Brazil", "Germany", "Argentina", "France", "Portugal", "Spain", "England", "Netherlands",
    "Uruguay", "Italy", "Belgium", "Croatia", "Morocco", "Japan", "Colombia", "Senegal",
    "USA", "Mexico", "Canada", "Ecuador", "Chile", "South Korea", "Switzerland", "Denmark",
    "Nigeria", "Algeria", "Egypt", "Ghana", "Ukraine", "Sweden", "Norway", "Turkey"
]

# Roda o mata-mata direto a partir da entrada customizada
historico_copa = simular_arvore_eliminatoria(meu_mata_mata_customizado, modo="estocastico")


🏆 --- SIMULANDO: DEZESSEIS-AVOS DE FINAL (ROUND OF 32) ---
 🟢 Brazil derrotou Germany no tempo normal.
 🟢 Argentina derrotou France no tempo normal.
 🟢 Portugal derrotou Spain no tempo normal.
 ⚖️ England vs Netherlands empataram no tempo regulamentar. [Netherlands] avança nos pênaltis!
 🟢 Uruguay derrotou Italy no tempo normal.
 🟢 Belgium derrotou Croatia no tempo normal.
 ⚖️ Morocco vs Japan empataram no tempo regulamentar. [Japan] avança nos pênaltis!
 🟢 Colombia derrotou Senegal no tempo normal.
 🟢 Mexico derrotou USA no tempo normal.
 🟢 Ecuador derrotou Canada no tempo normal.
 🟢 Chile derrotou South Korea no tempo normal.
 ⚖️ Switzerland vs Denmark empataram no tempo regulamentar. [Switzerland] avança nos pênaltis!
 🟢 Algeria derrotou Nigeria no tempo normal.
 🟢 Ghana derrotou Egypt no tempo normal.
 🟢 Sweden derrotou Ukraine no tempo normal.
 🟢 Norway derrotou Turkey no tempo normal.

🏆 --- SIMULANDO: OITAVAS DE FINAL ---
 🟢 Argentina derrotou Brazil no tempo normal.
 🟢 Portuga